# Kasiski Examination and Vigenère Cryptanalysis

This notebook is the executable companion to [Kasiski Test.md](Kasiski%20Test.md). It preserves the original path from Vigenère encryption to period discovery, frequency analysis, and decryption, while importing the corrected implementation from `kasiski.py`.

## 1. Alphabet and Vigenère arithmetic

With `A=0`, ..., `Z=25`, encryption is $C_i=P_i+K_{i\bmod\ell}\pmod{26}$ and decryption is $P_i=C_i-K_{i\bmod\ell}\pmod{26}$.

In [ ]:
from pathlib import Path

from kasiski import (
    ENGLISH_IC, RANDOM_IC, average_column_ic, break_vigenere,
    clean_text, factor_score, find_repeated_ngrams_positions,
    friedman_period_estimate, index_of_coincidence, kasiski_distances,
    rank_periods, rank_shifts_for_column, recover_key, split_columns,
    vigenere_decrypt, vigenere_encrypt,
)

ciphertext = vigenere_encrypt('ATTACK AT DAWN', 'LEMON')
assert ciphertext == 'LXFOPVEFRNHR'
assert vigenere_decrypt(ciphertext, 'LEMON') == 'ATTACKATDAWN'
ciphertext

## 2. Build the reproducible sample

The bundled source is normalized once and encrypted under the repeating key `MOUSE`. The attack below uses only the ciphertext and a public English-frequency model.

In [ ]:
plaintext = clean_text(Path('input-1-50.txt').read_text(encoding='utf-8'))
true_key = 'MOUSE'
ciphertext = vigenere_encrypt(plaintext, true_key)
print('normalized length:', len(ciphertext))
print('ciphertext prefix:', ciphertext[:80])

## 3. Repeated n-grams and spacings

If the same plaintext fragment is repeated at positions separated by a multiple of the key period, both copies meet the same key segment and normally produce the same ciphertext fragment. Accidental repetitions remain possible.

In [ ]:
repeated_trigrams = find_repeated_ngrams_positions(ciphertext, 3)
print('number of repeated trigrams:', len(repeated_trigrams))
print('sample:', list(repeated_trigrams.items())[:8])

In [ ]:
distance_counts, distances = kasiski_distances(ciphertext, min_n=3, max_n=5)
votes = factor_score(distances, max_period=20)
print('top distances:', distance_counts.most_common(10))
print('top divisor votes:', votes.most_common(10))

## 4. Index of Coincidence

For counts $f_x$ in a string of length $N$, $IC=\sum_x f_x(f_x-1)/(N(N-1))$. Uniform random letters have expected IC $1/26\approx0.03846$; the bundled English model has $\sum_xp_x^2\approx0.06550$. Correct periods and their multiples tend to produce language-like column IC.

In [ ]:
print('random baseline:', RANDOM_IC)
print('English model:', ENGLISH_IC)
print('full ciphertext IC:', index_of_coincidence(ciphertext))
for period in range(1, 21):
    print(f'{period:2d}: {average_column_ic(ciphertext, period):.4f}')

## 5. Combined period ranking

Kasiski votes can favour divisors; IC can favour multiples. The checked implementation retains both normalized components and combines them with an explicit teaching weight. It also reports Friedman's rough scalar estimate as a cross-check, not as ground truth.

In [ ]:
ranking = rank_periods(ciphertext, max_period=20)
for row in ranking[:10]:
    print(
        f'period={row.period:2d} votes={row.kasiski_votes:4d} '
        f'avgIC={row.average_ic:.4f} score={row.combined_score:.3f}'
    )
print('Friedman estimate:', friedman_period_estimate(ciphertext))
assert ranking[0].period == 5

## 6. Recover each Caesar shift

For every candidate shift in each column, decrypt the counts and minimize Pearson's chi-squared statistic against English monogram probabilities. This uses all letters and does not inspect the unknown plaintext.

In [ ]:
period = ranking[0].period
columns = split_columns(ciphertext, period)
for index, column in enumerate(columns):
    top = rank_shifts_for_column(column)[:3]
    print(index, [(shift, round(score, 2)) for shift, score in top])

recovered_key = recover_key(ciphertext, period)
print('recovered key:', recovered_key)
assert recovered_key == true_key

## 7. End-to-end attack and validation

The result retains the complete period ranking and distance histogram so the conclusion can be audited.

In [ ]:
result = break_vigenere(ciphertext, max_period=20)
print('period:', result.period)
print('key:', result.key)
print('plaintext prefix:', result.plaintext[:100])
assert result.period == 5
assert result.key == 'MOUSE'
assert result.plaintext == plaintext

## 8. Interpretation and next steps

A correct plaintext does not by itself prove a minimal period: a multiple can recover a repeated key. For shorter or noisier samples, retain several periods and several shifts per column, then use n-gram likelihood and beam search. Run `python -m unittest -v test_kasiski.py` for the complete regression suite.